# UC4 — Colab eğitim not defteri

Bu not defteri **yalnızca eğitim** adımlarını çalıştırır (01 → 04).
Quantization (05), final test değerlendirmesi (06) ve uygulama **laptopta, CPU'da** çalıştırılır.

**Kurallar**
- Test seti bu not defterinde **hiç kullanılmaz**. Test yalnızca `06_benchmark.py` içinde, laptopta, bir kez.
- Her hücreyi sırayla çalıştırın. Hata çıkarsa çıktının tamamını kopyalayın, düzeltelim.
- `--limit` ile yapılan deneme çalıştırmalarının sonuçları **raporlanmaz**.

**Hazırlık:** `us4` klasörünü (venv olmadan) Google Drive'ınıza `MyDrive/us4` olarak yükleyin.
Colab menüsü: *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*.

## 1. GPU kontrolü

In [ ]:
!nvidia-smi
import torch, platform
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU YOK")
print("python", platform.python_version())

## 2. Proje dosyaları
**Yöntem A (varsayılan): zip yükleme.** Laptopta `us4` klasörünü `us4.zip` olarak sıkıştırın,
sol menüdeki **Dosyalar** panelinden yükleyin, sonra bu hücreyi çalıştırın.
⚠️ Bu yöntemde dosyalar oturum kapanınca silinir; eğitim bitince 10. adımdaki zip'i hemen indirin.

**Yöntem B: Google Drive.** `USE_DRIVE = True` yapın (klasör `MyDrive/us4` olmalı).

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/us4'
else:
    import os
    assert os.path.exists('/content/us4.zip'), "us4.zip bulunamadı: Dosyalar panelinden /content içine yükleyin"
    !unzip -q -o /content/us4.zip -d /content/
    PROJECT_DIR = '/content/us4'

%cd {PROJECT_DIR}
!ls

## 3. Paketler
Kurulan sürümler `results/env_colab.txt` dosyasına kaydedilir (tekrarlanabilirlik).

In [ ]:
!pip install -q -r requirements-train.txt
!mkdir -p results && pip freeze > results/env_colab.txt
!python -c "import torch, transformers, gensim, sklearn, rapidfuzz; print('torch', torch.__version__, '| transformers', transformers.__version__, '| gensim', gensim.__version__, '| sklearn', sklearn.__version__, '| rapidfuzz', rapidfuzz.__version__)"

## 4. Birim testleri (normalizer + eşleştirici)

In [ ]:
!python -m pytest tests -q

## 5. Veri hazırlama (01)
İndirir, `data/processed/{train,dev,test}.tsv` olarak kaydeder, `results/data_report.json` yazar.

In [ ]:
!python src/01_data_prep.py

## 6. Sözlük oluşturma (Alan B, kural baseline)
Yalnızca train + dev kullanır. Çıktılar: `src/lexicon/*.txt`, `lexicon_meta.json`, `results/lexicon/*.csv`.

In [ ]:
!python src/lexicon/build_lexicon.py

## 7. TF-IDF + Lojistik Regresyon (02)

In [ ]:
!python src/02_classical_baseline.py

## 8. Word2Vec + sinir ağı (03)
İsteğe bağlı hızlı deneme (sonucu raporlanmaz, sonraki hücre üzerine yazar):
`!python src/03_word2vec_nn.py --limit 3000 --max-epochs 2`

In [ ]:
!python src/03_word2vec_nn.py

## 9. BERTurk ince ayarı (04)
Önce 1–2 dakikalık deneme önerilir (sonucu raporlanmaz; tam eğitim üzerine yazar):

`!python src/04_bertturk_finetune.py --limit 500 --epochs 1`

VRAM hatası (CUDA out of memory) olursa: `--batch 8 --accum 4`

In [ ]:
!python src/04_bertturk_finetune.py

## 10. Laptop için paketle ve indir
Zip'i laptoptaki `us4` klasörüne açın (aynı yollara). Zip boyutu birkaç yüz MB olabilir.

In [ ]:
!zip -qr colab_outputs.zip models src/lexicon/offensive_words.txt src/lexicon/safe_words.txt src/lexicon/known_words.txt src/lexicon/lexicon_meta.json results data/processed
!ls -lh colab_outputs.zip
!grep -h '"test_set_used"' models/*/meta.json src/lexicon/lexicon_meta.json
from google.colab import files
files.download('colab_outputs.zip')

## Sonraki adımlar (laptopta, CPU)
```powershell
python src/05_quantization.py
python src/06_benchmark.py
python app.py
```